## Multi-head Committee

By having multiple heads, we create a committee of experts:

- Head 1: Focuses on Grammar.
- Head 2: Focuses on Adjectives.
- Head 3: Focuses on Context.
- ...and so on.

### How do Heads decide what to learn?

You asked: How does a head know it should be the Grammar Head or the Spelling Head?

The absolute truth: We do not tell them what to do. There is no code that says if head == 1: learn_grammar().

They learn their jobs through Emergence and Symmetry Breaking. Here is how it happens during training:

#### 1. Random Beginnings (Step 0)

When you initialize the model, the weights ($W_q, W_k, W_v$) for all 8 heads are filled with random numbers.
Head 1 is random garbage.
Head 2 is slightly different random garbage.

#### 2. The First Forward Pass

You pass in "The cat sat on the mat." The network makes a terrible prediction because it's random. The Loss function says, "You are 99% wrong."

#### 3. Backpropagation (The Coach)

The math of Backpropagation calculates the gradients: "How should we change the weights to be slightly less wrong next time?"
Because Head 1 and Head 2 started with different random numbers, they made slightly different mistakes.
Let's say Head 1 accidentally paid 5% more attention to the verb "sat". Backprop notices: "Hey, looking at verbs helped reduce the error! Head 1, keep doing that, but more."
Head 2 accidentally paid 5% more attention to the adjective. Backprop says: "Head 2, looking at adjectives helped! Do more of that."

#### 4. Symmetry Breaking (Avoid Redundancy)

If Head 1 becomes a perfect "Grammar Head," there is no incentive for Head 2 to also become a Grammar Head. Why? Because if Head 2 copies Head 1, the network doesn't gain any new information, so the Loss doesn't go down.
To minimize the Loss, the math actively pushes the heads to find different, unique patterns.
Over billions of training steps, this forces them to specialize:

- Positional Heads: One head learns to just look at the word immediately to the left.
- Syntax Heads: One head learns to link pronouns ("he") to subjects ("John").
- Vocabulary Heads: One head learns to link related concepts ("king" -> "royal").

It is not hardcoded. It is the natural mathematical result of a team (the 8 heads) trying to get the highest score possible on a test (minimizing the loss) while starting from different random positions.

# Working of Multi-Heads in Parallel

Think of it like a hierarchy of boxes:

- **The Big Box (Batch):** The entire sentence ("Hi there").
- **The Medium Boxes (Tokens/Words):** The individual words inside the sentence ("Hi", "there").
- **The Small Items (Embeddings):** The numbers inside each word that describe its meaning.

Let's build this properly.

### The Real Data Structure

We have 1 Sentence (Batch = 1). It has 2 Words (Tokens = 2: "Hi", "there"). Each word is described by 4 Numbers (Embedding = 4).

Here is exactly how PyTorch stores your sentence in memory:

```python
[  # <--- START OF BATCH (The Sentence: "Hi there")
    
    [1, 2, 3, 4],  # <--- Word 1 ("Hi")
    [5, 6, 7, 8]   # <--- Word 2 ("there")

]  # <--- END OF BATCH
```

The shape of this data is `(1, 2, 4)` -> `(Batch, Words, Numbers)`.

### The Goal

We want 2 Attention Heads. This means we need to chop the 4 numbers of every word in half.

- **Head 1** gets the first 2 numbers of every word.
- **Head 2** gets the last 2 numbers of every word.

We need PyTorch to reorganize the boxes so that Head 1 and Head 2 have their own isolated workspaces, completely separated from each other, but still keeping the words in order.

### Step 1: The `.view()` Command (The Slice)

**Code:** `q.view(Batch=1, Words=2, Heads=2, Head_Size=2)`

We tell PyTorch: "Go inside every Word box, and slice those 4 numbers into 2 separate Head boxes."

Watch the brackets change. PyTorch literally cuts `[1, 2, 3, 4]` in half:

```python
[  # START OF BATCH (Sentence)
    
    # Word 1 ("Hi") is sliced:
    [ 
      [1, 2],  # <-- Piece for Head 1
      [3, 4]   # <-- Piece for Head 2
    ],
    
    # Word 2 ("there") is sliced:
    [ 
      [5, 6],  # <-- Piece for Head 1
      [7, 8]   # <-- Piece for Head 2
    ]

]
```

The shape is now `(1, 2, 2, 2)` -> `(Batch, Words, Heads, Numbers)`.

**Why we aren't done:** Look at the data. The numbers are sliced, but they are still trapped inside the "Word" boxes. Head 1's numbers (`[1, 2]` and `[5, 6]`) are separated from each other. We need to group them by Head, not by Word.

### Step 2: The `.transpose(1, 2)` Command (The Swap)

This is the magic move. Transpose means "swap the order of the boxes." We tell PyTorch: "Swap the 'Words' box (dimension 1) with the 'Heads' box (dimension 2)."

Instead of "Words containing Heads", we want "Heads containing Words".

Watch how PyTorch regroups the exact same numbers:

```python
[  # START OF BATCH (Sentence)

    [  # <--- START OF HEAD 1'S WORKSPACE
        [1, 2],  # Word 1 ("Hi")
        [5, 6]   # Word 2 ("there")
    ],

    [  # <--- START OF HEAD 2'S WORKSPACE
        [3, 4],  # Word 1 ("Hi")
        [7, 8]   # Word 2 ("there")
    ]

]
```

The final shape is `(1, 2, 2, 2)` -> `(Batch, Heads, Words, Numbers)`.

### Why this is the breakthrough

Now, Head 1 has a perfect, isolated list of all the words (just the first half of their meanings). Head 2 has its own isolated list.

When you pass this to the GPU to do the Attention Math, the GPU says: "Great! I have two independent workspaces. I'll calculate Head 1's attention on my left chip, and Head 2's attention on my right chip, at the exact same time."

# Doesnt chopping cut off data?

### The Problem: Chopping Raw Data
Imagine our word "Apple" is represented by 4 numbers. Let's pretend each number means something specific:

```text
[ 10,  20,  30,  40 ]
```

* **10** = Grammar info
* **20** = Tone info
* **30** = Definition info
* **40** = Context info

If we just take a knife and slice this array in half right now:

* **Head 1** gets `[10, 20]` (Only Grammar and Tone). It is completely blind to Definition and Context.
* **Head 2** gets `[30, 40]` (Only Definition and Context).

If we did this, the model would be broken.

### The Solution: The Linear Layer (The "Blender")
Before we slice anything, we pass that `[10, 20, 30, 40]` through a Linear Layer.

A Linear Layer is literally just a giant web of multiplication and addition. In deep learning, we call it a "Fully Connected" layer. "Fully connected" means every single output number is connected to every single input number.

Let's look at the actual math of how the Linear Layer creates its first new output number (let's call it `New_Number_1`).

The layer has learned "weights" (multipliers). It takes a little bit of everything from the original word:

* $(0.5 \times 10 \text{ Grammar}) = 5$
* $(0.1 \times 20 \text{ Tone}) = 2$
* $(0.8 \times 30 \text{ Definition}) = 24$
* $(0.2 \times 40 \text{ Context}) = 8$

Add them all together: $5 + 2 + 24 + 8 = 39$

So, `New_Number_1` is **39**.

Look closely at what just happened. That **39** is not just "grammar." It is a mathematical smoothie. It contains a percentage of grammar, tone, definition, and context, all baked into one single number.

The Linear Layer does this 4 times to create a brand new 4-number array:
```text
[ 39,  15,  88,  42 ]
```

### The Magic of the Slice
Now we take our knife and slice the array for our 2 Heads.

* **Head 1** gets `[39, 15]`.
* **Head 2** gets `[88, 42]`.

**Does Head 1 lose the big picture?**
Absolutely not. Look at **39**. We just proved that **39** contains information about the Grammar, Tone, Definition, and Context.

By passing the data through the Linear Layer before we slice it, we ensured that every slice contains a compressed summary of the entire original word.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads # 512 / 8 = 64
        
        # 1. The Pre-Slice Blenders (No data is lost!)
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        
        # 2. The Final Blender (Sharing notes)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        print(f"[Step 0] Input Shape:         {x.shape} -> (Batch, Tokens, Dim)")
        
        # --- 1. The Blender ---
        # Mix the 512 raw ingredients into a new 512 smoothie.
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)
        print(f"[Step 1] After Blender:       {q.shape} -> (Batch, Tokens, Dim)")
        
        # --- 2. The Slice (.view) ---
        # Cut the 512 smoothie into 8 cups of 64.
        q = q.view(B, T, self.num_heads, self.head_dim)
        k = k.view(B, T, self.num_heads, self.head_dim)
        v = v.view(B, T, self.num_heads, self.head_dim)
        print(f"[Step 2] After Slice:         {q.shape} -> (Batch, Tokens, Heads, Head_Dim)")
        
        # --- 3. The Swap (.transpose) ---
        # Move the 'Heads' box to the outside so the GPU can parallelize them.
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        print(f"[Step 3] After Swap:          {q.shape} -> (Batch, Heads, Tokens, Head_Dim)")
        
        # --- 4. The Attention Math ---
        # The GPU sees (Tokens, 64) @ (64, Tokens) and does it 8 times at once.
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal Mask (No peeking at the future)
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Probabilities
        attn_weights = F.softmax(scores, dim=-1)
        print(f"[Step 4] Attention Grid:      {attn_weights.shape} -> (Batch, Heads, Tokens, Tokens)")
        
        # Weighted retrieval of Values
        out = torch.matmul(attn_weights, v)
        print(f"[Step 5] Head Outputs:        {out.shape} -> (Batch, Heads, Tokens, Head_Dim)")
        
        # --- 5. The Reassembly ---
        # Un-swap the heads back to the inside.
        out = out.transpose(1, 2).contiguous()
        print(f"[Step 6] After Un-Swap:       {out.shape} -> (Batch, Tokens, Heads, Head_Dim)")
        
        # Smush the 8 cups of 64 back into a single 512 block.
        out = out.view(B, T, self.d_model)
        print(f"[Step 7] After Smush:         {out.shape} -> (Batch, Tokens, Dim)")
        
        # --- 6. The Final Blender ---
        # Mix the findings of all 8 heads together.
        out = self.w_o(out)
        print(f"[Step 8] Final Output:        {out.shape} -> (Batch, Tokens, Dim)")
        
        return out

# --- Execute the Lab ---
if __name__ == "__main__":
    # Create a fake sentence: 1 Batch, 5 Words, 512 Dimension
    dummy_sentence = torch.randn(1, 5, 512)
    
    # Initialize our engine
    mha = MultiHeadAttention(d_model=512, num_heads=8)
    
    # Run it!
    print("--- RUNNING MHA TENSOR TRACE ---\n")
    final_result = mha(dummy_sentence)
    print("\n✅ SUCCESS: The word entered as 512 and left as a context-aware 512.")

--- RUNNING MHA TENSOR TRACE ---

[Step 0] Input Shape:         torch.Size([1, 5, 512]) -> (Batch, Tokens, Dim)
[Step 1] After Blender:       torch.Size([1, 5, 512]) -> (Batch, Tokens, Dim)
[Step 2] After Slice:         torch.Size([1, 5, 8, 64]) -> (Batch, Tokens, Heads, Head_Dim)
[Step 3] After Swap:          torch.Size([1, 8, 5, 64]) -> (Batch, Heads, Tokens, Head_Dim)
[Step 4] Attention Grid:      torch.Size([1, 8, 5, 5]) -> (Batch, Heads, Tokens, Tokens)
[Step 5] Head Outputs:        torch.Size([1, 8, 5, 64]) -> (Batch, Heads, Tokens, Head_Dim)
[Step 6] After Un-Swap:       torch.Size([1, 5, 8, 64]) -> (Batch, Tokens, Heads, Head_Dim)
[Step 7] After Smush:         torch.Size([1, 5, 512]) -> (Batch, Tokens, Dim)
[Step 8] Final Output:        torch.Size([1, 5, 512]) -> (Batch, Tokens, Dim)

✅ SUCCESS: The word entered as 512 and left as a context-aware 512.


Here is the absolute, no-nonsense summary of the Multi-Head Attention (MHA) flow, step-by-step:

### The Multi-Head Attention Pipeline

**1. The Input**
* You start with a batch of whole word embeddings.
* **Shape:** `(Batch, Tokens, 512)`

**2. The Initial Projections (The "Blender")**
* The input is copied and passed through three separate Linear layers ($W_q$, $W_k$, $W_v$).
* This mixes the raw data into three new vectors: Query, Key, and Value. No data is lost; it's just mathematically summarized.
* **Shape remains:** `(Batch, Tokens, 512)`

**3. The Slice (`.view`)**
* The 512-dimension vectors are chopped into multiple smaller pieces (e.g., 8 heads, each with 64 dimensions).
* **Shape becomes:** `(Batch, Tokens, 8 Heads, 64)`

**4. The Swap (`.transpose`)**
* The "Tokens" and "Heads" dimensions are swapped. This groups the data by Head so the GPU can run them all in parallel.
* **Shape becomes:** `(Batch, 8 Heads, Tokens, 64)`

**5. Parallel Attention Math (The Core Logic)**
* **Dot Product:** $Q \times K^T$ (Calculates raw similarity scores).
* **Scale:** Divide by $\sqrt{64}$ (Keeps gradients stable).
* **Mask:** Apply the causal lower-triangle mask (Blocks peeking at the future).
* **Softmax:** Convert scores to percentages (Sums to 100%).
* **Multiply:** Multiply the percentages by $V$ (Retrieves the actual contextual information).
* **Shape remains:** `(Batch, 8 Heads, Tokens, 64)`

**6. The Reassembly (Un-swap and "Smush")**
* Swap the "Tokens" and "Heads" back to their original order.
* Concatenate (smush) the 8 separate 64-dimension chunks back into a single 512-dimension vector.
* **Shape returns to:** `(Batch, Tokens, 512)`

**7. The Final Projection (The Output "Blender")**
* Pass the reassembled 512-vector through one last Linear layer ($W_o$).
* This forces the 8 separate heads to share their unique findings with each other, creating a single, highly intelligent token.
* **Final Shape:** `(Batch, Tokens, 512)`